# Paper 6 (RQ-F7) — Algorithmic concentration, fragility, phase transition

Sweeps adversarial market share across the defended arena to locate the concentration threshold at which microstructure defenses break down, and compares monoculture vs diversity.

Self-contained Colab notebook. **Runtime -> Run all**, or run cells top to bottom.
Clones the repo, installs the package, runs the scenarios this paper needs, and saves
the result CSVs to Google Drive (or Supabase).


## 1. Clone the repo and install


In [ ]:
REPO = 'https://github.com/zalliata/marketsim-v2.git'
# Private repo? Uncomment and paste a GitHub token (Contents: read):
# TOKEN = 'ghp_xxx'; REPO = REPO.replace('https://', f'https://{TOKEN}@')
import os, shutil
if os.path.exists('marketsim-v2'): shutil.rmtree('marketsim-v2')
!git clone -q $REPO
%cd marketsim-v2
!pip install -q -e .
print('installed:', os.path.exists('pyproject.toml'))


## 2. Sanity check


In [ ]:
!python -m pytest -q


## 3. (Optional) Save results to Google Drive
Skip to keep results only in the temporary Colab filesystem.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/marketsim-results', exist_ok=True)
print('Drive mounted')


## 4. (Optional) Write to Supabase instead of local CSV
Leave unrun for local CSV (default). Prefer Colab Secrets over pasting keys.


In [ ]:
import os
# os.environ['SUPABASE_URL'] = 'https://xxxx.supabase.co'
# os.environ['SUPABASE_SERVICE_KEY'] = 'your-service-key'
# !pip install -q supabase
print('Supabase configured:', bool(os.environ.get('SUPABASE_URL')))


## 5. Shared runner


In [ ]:
from marketsim.scenarios import definitions as D
from marketsim.runner.batch import run_batch, run_sweep
from marketsim.storage import make_backend

ITER = 20         # per condition; set to 100 for the reported runs
LLM  = 'scripted' # deterministic adversary (free). 'anthropic' etc. only for tiny batches

def run_scenarios(ids, iterations=None, llm=None):
    iterations = iterations or ITER; llm = llm or LLM
    backend = make_backend('auto')
    try:
        for sid in ids:
            sc = D.get(sid)
            if sc.sweep:
                stats = run_sweep(sc, iterations, backend, llm_mode=llm)
                print(f'[{sid}] sweep {sc.sweep.kind} - {len(stats)} points:')
                for st in stats:
                    print(f'    {st.grid_key}={st.grid_value:<7} vol={st.vol_mean:.4f} '
                          f'adv_pnl={st.adversary_pnl_mean:>12,.0f} cascade={st.cascade_freq_mean:.3f} '
                          f'stab={st.stabilisation_mean:.3f}')
            else:
                st = run_batch(sc, iterations, backend, llm_mode=llm)
                print(f'[{sid}] batch n={st.n}: vol={st.vol_mean:.4f} '
                      f'adv_pnl={st.adversary_pnl_mean:,.0f} cascade={st.cascade_freq_mean:.3f}')
    finally:
        backend.close()
    print('--- done ---')


## 6. Composition sweep
Adversarial share 5% -> 50% across the full defended arena. The cascade-frequency and
stabilisation-effectiveness columns trace the degradation curve and locate the phase
transition (H1: a discontinuous jump between ~25-40%).


In [ ]:
run_scenarios(['p6-composition-sweep'], iterations=25)   # 10 share points; raise once timed


## 7. Monoculture vs diversity
At each share level: a single dominant adversary strategy vs a mixed adversary population.
Tests H2 (strategy diversity delays the phase transition relative to monoculture).


In [ ]:
run_scenarios(['p6-monoculture', 'p6-diversity'], iterations=25)


## 8. Locate the phase transition
Turns the composition sweep's stabilisation and cascade curves into a threshold estimate
(the share where stabilisation drops below half its low-share value).


In [ ]:
from marketsim.metrics.systemic import locate_phase_transition
backend = make_backend('local')
try:
    stats = run_sweep(D.get('p6-composition-sweep'), ITER, backend)
finally:
    backend.close()

shares = [s.grid_value for s in stats]
stab   = [s.stabilisation_mean for s in stats]
casc   = [s.cascade_freq_mean for s in stats]
pt = locate_phase_transition(shares, stab, casc)
print('phase transition found:', pt.found, '| threshold adversary share =', pt.threshold_share)
for sh, st in pt.stabilisation_curve:
    print(f'  share={sh:<6} stabilisation={st:.3f}')


## 9. Plot the degradation curve


In [ ]:
import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(6,4))
ax1.plot(shares, casc, 'o-', color='crimson', label='cascade frequency')
ax1.set_xlabel('adversarial market share'); ax1.set_ylabel('cascade frequency', color='crimson')
ax2 = ax1.twinx()
ax2.plot(shares, stab, 's--', color='navy', label='stabilisation effectiveness')
ax2.set_ylabel('stabilisation effectiveness', color='navy')
if pt.found: ax1.axvline(pt.threshold_share, color='grey', ls=':', label='phase transition')
fig.suptitle('P6: fragility vs algorithmic concentration'); fig.tight_layout(); plt.show()


## Collect results
CSVs land in `results/<run_id>/`. Copy to Drive (if mounted) or zip and download.


In [ ]:
import glob, os, shutil
runs = sorted(glob.glob('results/*'))
print(len(runs), 'run folders')
if os.path.exists('/content/drive/MyDrive'):
    dest = '/content/drive/MyDrive/marketsim-results'
    for r in runs:
        shutil.copytree(r, os.path.join(dest, os.path.basename(r)), dirs_exist_ok=True)
    print('copied to', dest)
else:
    shutil.make_archive('marketsim_results', 'zip', 'results')
    from google.colab import files; files.download('marketsim_results.zip')


---
### Notes
- These are the heaviest jobs: each sweep runs the battery at every share level. Start at
  `iterations=25`; for the reported run use 100 and consider splitting the share grid across
  sessions if Colab times out.
- The composition sweep's `adversary_share` column keys the phase-transition analysis; the
  monoculture/diversity comparison tests the diversity hypothesis.
- Ask Claude to build the full phase-transition writeup + policy-threshold calc when ready.
